# Day-ahead purchase volume under imbalance prices — corrected

Each fix is marked **Fix N** (numbers refer to `mock_13_solution.md`).

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, QuantileRegressor

pd.set_option("display.width", 120)

In [2]:
df = pd.read_csv("../../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
SHARE = 0.01
rng = np.random.default_rng(42)
n = len(df)
premium = rng.lognormal(np.log(25) - 0.5 * 0.6**2, 0.6, n)
discount = rng.lognormal(np.log(15) - 0.5 * 0.6**2, 0.6, n)
imb = pd.DataFrame({"da_price": df["price_eur_mwh"],
                    "sys_buy": df["price_eur_mwh"] + premium,
                    "sys_sell": df["price_eur_mwh"] - discount}, index=df.index)

**Fix 9 (timezones).** Keep every frame in UTC and join on the UTC index. The original converted
the price frame to London wall-clock and dropped the zone, then joined it to a UTC-naive load frame:
every BST hour was matched to the wrong price, one autumn hour was duplicated and one spring hour
lost.

In [3]:
loc = df.index.tz_convert("Europe/London").tz_localize(None)
print("duplicated local wall-clock stamps:", loc.duplicated().sum(),
      "| hours where local != UTC wall clock:", (loc != df.index.tz_localize(None)).sum())

duplicated local wall-clock stamps: 2 | hours where local != UTC wall clock: 10416


## Load forecast

**Fix 2 (fit on train only).** The model is fitted on 2022 and evaluated on 2023. The residual
distribution used for the safety margin comes from the training residuals of that model.

In [4]:
cons = df["consumption_mwh"]
X = pd.DataFrame({"lag24": cons.shift(24), "lag168": cons.shift(168), "temp": df["temp_c"],
                  "hour": df.index.hour, "dow": df.index.dayofweek})
X = pd.get_dummies(X, columns=["hour", "dow"], dtype=float)
data = X.assign(y=cons).dropna()
features = [c for c in data.columns if c != "y"]
train, test = data.loc["2022"].copy(), data.loc["2023"].copy()

model = Ridge(alpha=1.0).fit(train[features], train["y"])
train["fc"] = model.predict(train[features]); test["fc"] = model.predict(test[features])
train["resid"] = train["y"] - train["fc"]; test["resid"] = test["y"] - test["fc"]
print(f"train resid sd {train['resid'].std():.0f} MWh, test resid sd {test['resid'].std():.0f} MWh")

train resid sd 897 MWh, test resid sd 894 MWh


## Optimal purchase quantile

**Fix 1 (critical fractile).** Being *short* costs the premium `c_u` ≈ 25; being *long* costs the
discount `c_o` ≈ 15. The newsvendor optimum is the `c_u / (c_u + c_o)` quantile: 0.62, i.e. buy
*above* the median forecast. The original used `c_o / (c_u + c_o)` = 0.38 and bought below it.

**Fix 4 (units).** The residuals are in MWh of *national* load. Convert to the book's share with
`SHARE`, not with an ad-hoc `/ 100` two cells later (which happened to be the same number, and the
unconverted −285 MWh still went into the summary table as the "safety margin").

In [5]:
c_u, c_o = premium.mean(), discount.mean()
q_star = c_u / (c_u + c_o)
adj_national = np.quantile(train["resid"], q_star)
adj = adj_national * SHARE
print(f"c_u = {c_u:.1f}, c_o = {c_o:.1f}, q* = {q_star:.3f}")
print(f"safety margin: {adj_national:+.0f} MWh national = {adj:+.2f} MWh for the book (~{adj / (test['fc'].mean()*SHARE) * 100:+.1f}% of forecast)")

c_u = 25.0, c_o = 15.2, q* = 0.622
safety margin: +264 MWh national = +2.64 MWh for the book (~+0.9% of forecast)


In [6]:
book = pd.DataFrame({"load": test["y"] * SHARE, "fc": test["fc"] * SHARE})
book["q"] = book["fc"] + adj
book = book.join(imb, how="inner")
print(len(book), "hours (all of 2023 after the lag warm-up)")

8760 hours (all of 2023 after the lag warm-up)


## Imbalance cost

**Fix 5 (cost function sides).** Short is `load > q`, i.e. `max(load - q, 0)`, charged at the
premium; long is `max(q - load, 0)`, charged at the discount. The original had them swapped, which
made buying less look cheaper and "confirmed" the inverted fractile.

**Fix 6 (both sides).** Total imbalance cost is over *all* hours. Summing only the hours where we
were short discards the long-side cost entirely.

In [7]:
def imbalance_cost(q, load, sys_buy, sys_sell, da):
    short = np.maximum(load - q, 0)
    long_ = np.maximum(q - load, 0)
    return short * (sys_buy - da) + long_ * (da - sys_sell)

def total(q):
    return imbalance_cost(q, book["load"], book["sys_buy"], book["sys_sell"], book["da_price"]).sum()

da_cost = (book["fc"] * book["da_price"]).sum()
print(f"day-ahead purchase cost: {da_cost/1e6:,.1f} mEUR; imbalance cost at forecast: {total(book['fc'])/1e3:,.0f} kEUR "
      f"({total(book['fc'])/da_cost*100:.2f}% of procurement)")

day-ahead purchase cost: 226.7 mEUR; imbalance cost at forecast: 1,203 kEUR (0.53% of procurement)


**Fix 3 (tune on train, test on test).** Any margin chosen by minimising 2023 cost and then
reported on 2023 is in-sample. Choose it on 2022 (the model's own training residuals with the same
imbalance-price mechanics), then apply once to 2023.

In [8]:
tb = pd.DataFrame({"load": train["y"] * SHARE, "fc": train["fc"] * SHARE}).join(imb, how="inner")
def total_train(q):
    return imbalance_cost(q, tb["load"], tb["sys_buy"], tb["sys_sell"], tb["da_price"]).sum()

grid = np.arange(-0.05, 0.051, 0.005)
tr_cost = pd.Series({round(m, 3): total_train(tb["fc"] * (1 + m)) for m in grid})
best_margin = tr_cost.idxmin()
te_cost = pd.Series({round(m, 3): total(book["fc"] * (1 + m)) for m in grid})
print(f"margin chosen on 2022: {best_margin:+.1%}")
pd.DataFrame({"2022 cost (kEUR)": tr_cost / 1e3, "2023 cost (kEUR)": te_cost / 1e3}).round(0).loc[[-0.02, -0.01, 0.0, 0.005, 0.01, 0.02, 0.03]]

margin chosen on 2022: +1.0%


,2022 cost (kEUR),2023 cost (kEUR)
-0.020,1739.0,1618.0
-0.010,1423.0,1339.0
-0.000,1231.0,1203.0
0.005,1189.0,1187.0
0.010,1181.0,1203.0
0.020,1261.0,1326.0
0.030,1441.0,1540.0


**Fix 8 (quantile regression, right quantile, right metric).** A quantile model should be fitted
at the decision quantile (0.62, not 0.9) and judged by pinball loss and realised imbalance cost.
RMSE rewards the mean; a 0.9-quantile model is *supposed* to have a worse RMSE.

In [9]:
def pinball(y, q, tau):
    d = y - q
    return np.mean(np.maximum(tau * d, (tau - 1) * d))

qr = QuantileRegressor(quantile=q_star, alpha=0.0, solver="highs").fit(train[features], train["y"])
book["q_qr"] = qr.predict(test[features]) * SHARE
book["q_pct"] = book["fc"] * (1 + best_margin)

rows = []
for lab, q in [("buy forecast", book["fc"]), ("forecast + q* residual quantile", book["q"]),
               ("forecast * (1 + margin from 2022)", book["q_pct"]), ("quantile regression at q*", book["q_qr"]),
               ("original rule: forecast - 2%", book["fc"] * 0.98)]:
    rows.append({"rule": lab, "imbalance cost 2023 (kEUR)": total(q) / 1e3,
                 "pinball@q*": pinball(book["load"], q, q_star),
                 "share of hours short": (book["load"] > q).mean()})
pd.DataFrame(rows).set_index("rule").round(3)

,imbalance cost 2023 (kEUR),pinball@q*,share of hours short
rule,,,
buy forecast,1203.078,3.422,0.439
forecast + q* residual quantile,1195.494,3.410,0.326
forecast * (1 + margin from 2022),1202.996,3.434,0.314
quantile regression at q*,1198.510,3.420,0.325
original rule: forecast - 2%,1617.501,4.583,0.707


**Fix 10 (Monte Carlo).** One generator created outside the loop. `np.random.seed(42)` *inside*
the loop made all 200 scenarios identical, which is why the original CI had zero width.

In [10]:
rng_mc = np.random.default_rng(7)
sims = []
for _ in range(200):
    r = rng_mc.choice(train["resid"].values, size=len(book), replace=True) * SHARE
    load_sim = book["fc"] + r
    sims.append(imbalance_cost(book["q"], load_sim, book["sys_buy"], book["sys_sell"], book["da_price"]).sum())
sims = np.array(sims)
print(f"expected cost {sims.mean()/1e3:,.0f} kEUR, 95% CI [{np.percentile(sims, 2.5)/1e3:,.0f}, {np.percentile(sims, 97.5)/1e3:,.0f}] kEUR")

expected cost 1,195 kEUR, 95% CI [1,166, 1,226] kEUR


**Fix 11 (ratios with prices near zero).** `cost / da_price` explodes when the day-ahead price is
near zero and flips sign when it is negative (29 hours ≤ 0 in 2023; the original's March average
was negative). If a normalised view is wanted, normalise by something that cannot be zero (here
the buy–sell spread), or exclude those hours and say so.

In [11]:
neg = (book["da_price"] <= 0).sum()
book["cost_q"] = imbalance_cost(book["q"], book["load"], book["sys_buy"], book["sys_sell"], book["da_price"])
ratio = book["cost_q"] / book["da_price"]
print(f"hours with da_price <= 0: {neg}")
print(f"naive ratio range: {ratio.min():,.0f} to {ratio.max():,.0f} (monthly means are meaningless)")
(book["cost_q"] / (book["sys_buy"] - book["sys_sell"])).resample("MS").mean().round(2).head(3)

hours with da_price <= 0: 29
naive ratio range: -750 to 253 (monthly means are meaningless)


time
2023-01-01 00:00:00+00:00    3.23
2023-02-01 00:00:00+00:00    3.44
2023-03-01 00:00:00+00:00    3.17
Freq: MS, dtype: float64

## Results

In [12]:
print(f"critical fractile q* = {q_star:.3f} -> buy ABOVE the median forecast")
print(f"imbalance cost 2023, buy forecast:      {total(book['fc'])/1e3:,.0f} kEUR")
print(f"imbalance cost 2023, q* residual rule:  {total(book['q'])/1e3:,.0f} kEUR")
print(f"imbalance cost 2023, quantile regression:{total(book['q_qr'])/1e3:,.0f} kEUR")
print(f"imbalance cost 2023, original -2% rule: {total(book['fc']*0.98)/1e3:,.0f} kEUR")
print(f"imbalance cost as share of procurement:  {total(book['fc'])/da_cost*100:.2f}%")

critical fractile q* = 0.622 -> buy ABOVE the median forecast
imbalance cost 2023, buy forecast:      1,203 kEUR
imbalance cost 2023, q* residual rule:  1,195 kEUR
imbalance cost 2023, quantile regression:1,199 kEUR
imbalance cost 2023, original -2% rule: 1,618 kEUR
imbalance cost as share of procurement:  0.53%


The direction of the original recommendation was wrong: under-buying by 1% costs 11% more than
buying the forecast, and by 2% 34% more. The correct newsvendor adjustment (buy slightly above the median) saves less
than 1% of imbalance cost here, because the forecast residuals are nearly symmetric and the
premium/discount asymmetry is mild. Imbalance is ~0.5% of procurement cost, so forecast accuracy
matters far more than the quantile rule. That is the honest answer, and it is worth saying out
loud instead of reporting a large saving.